# D2


In [1]:
!apt-get install -y -qq rabbitmq-server > /dev/null
!service rabbitmq-server start
!pip install -q pika

[ OK ]


In [2]:
!sleep 15
!rabbitmqctl status | head -8

Status of node rabbit@79d1b957d175 ...
Runtime

OS PID: 2731
OS: Linux
Uptime (seconds): 34
Is under maintenance?: false
RabbitMQ version: 3.12.1


In [3]:
!git clone -q https://github.com/OPHELIA113/freshcart-pipelines.git
%cd freshcart-pipelines
!ls stream

/content/freshcart-pipelines
dispatcher.py  driver.py


In [4]:
##Run the dispatcher and six drivers for two minutes

import subprocess, time, os
os.makedirs("logs", exist_ok=True)

drivers = [("Ravi", "bangalore.koramangala.*"), ("Priya", "bangalore.indiranagar.*"),
           ("Arjun", "bangalore.*.ride"), ("Meena", "bangalore.*.food"),
           ("Karthik", "chennai.tnagar.food"), ("Control", "bangalore.#")]

procs = []
for name, pattern in drivers:
    f = open(f"logs/{name}.log", "w")
    procs.append(subprocess.Popen(["python", "-u", "stream/driver.py", name, pattern],
                                  stdout=f, stderr=subprocess.STDOUT))
time.sleep(4)                                       # let every driver bind its queue

disp = subprocess.Popen(["python", "-u", "stream/dispatcher.py"],
                        stdout=open("logs/dispatcher.log", "w"), stderr=subprocess.STDOUT)

time.sleep(6)
# one extra message for D1 question 6 (a two-word key the random dispatcher never makes)
subprocess.run(["python", "-u", "stream/dispatcher.py", "bangalore.koramangala"],
               stdout=open("logs/oneshot.log", "w"), stderr=subprocess.STDOUT)

time.sleep(114)                                     # about two minutes in total
disp.terminate()
time.sleep(3)
for p in procs:
    p.terminate()
print("done")


done


In [5]:
!cat logs/Ravi.log

[Ravi] waiting with pattern 'bangalore.koramangala.*' on queue amq.gen-hpE_2ZuRHuDhlTq7LHNetA
[Ravi] key=bangalore.koramangala.food id=66bbf3c7 zone=koramangala type=food
[Ravi] key=bangalore.koramangala.ride id=4e23b3b6 zone=koramangala type=ride
[Ravi] key=bangalore.koramangala.ride id=08d79865 zone=koramangala type=ride
[Ravi] key=bangalore.koramangala.ride id=4d0266ce zone=koramangala type=ride
[Ravi] key=bangalore.koramangala.ride id=4e701751 zone=koramangala type=ride
[Ravi] key=bangalore.koramangala.ride id=f483c783 zone=koramangala type=ride
[Ravi] key=bangalore.koramangala.food id=2998b35c zone=koramangala type=food
[Ravi] key=bangalore.koramangala.food id=626a19c4 zone=koramangala type=food
[Ravi] key=bangalore.koramangala.ride id=11937b75 zone=koramangala type=ride
[Ravi] key=bangalore.koramangala.ride id=1214b782 zone=koramangala type=ride
[Ravi] key=bangalore.koramangala.food id=594e8c08 zone=koramangala type=food


In [7]:
!head -10 logs/dispatcher.log

published key=chennai.tnagar.food id=8b0d04a1
published key=chennai.whitefield.food id=8e3d1ca1
published key=bangalore.whitefield.food id=c6fd08b6
published key=chennai.whitefield.ride id=dac45fb1
published key=chennai.tnagar.food id=84897894
published key=chennai.tnagar.food id=25530377
published key=bangalore.whitefield.ride id=10534754
published key=chennai.tnagar.ride id=4ef509d2
published key=chennai.whitefield.food id=41ea928f
published key=chennai.whitefield.ride id=66aef696


In [6]:
##Compare what each driver received with my predictions

import re, glob

expected = {
    "bangalore.koramangala.food": {"Ravi", "Meena", "Control"},
    "bangalore.indiranagar.ride": {"Priya", "Arjun", "Control"},
    "bangalore.whitefield.ride":  {"Arjun", "Control"},
    "chennai.tnagar.food":        {"Karthik"},
    "bangalore.koramangala":      {"Control"},
}

published = set()
for f in ["logs/dispatcher.log", "logs/oneshot.log"]:
    published |= set(re.findall(r"published key=(\S+)", open(f).read()))

received = {}
for f in glob.glob("logs/*.log"):
    name = f.split("/")[-1][:-4]
    if name in ("dispatcher", "oneshot"):
        continue
    for key in re.findall(r"\]\s+key=(\S+)", open(f).read()):
        received.setdefault(key, set()).add(name)

print(f"{'routing key':32s} {'received by':32s} matches D1?")
for key in sorted(published):
    got = received.get(key, set())
    status = ("OK" if got == expected[key] else "DIFFERENT") if key in expected else "(not in D1 table)"
    print(f"{key:32s} {', '.join(sorted(got)) or 'dropped':32s} {status}")

print("\nD1 key 5 (chennai.whitefield.ride) published:", "chennai.whitefield.ride" in published,
      "| received by:", received.get("chennai.whitefield.ride", "nobody"))

routing key                      received by                      matches D1?
bangalore.indiranagar.food       Control, Meena, Priya            (not in D1 table)
bangalore.indiranagar.ride       Arjun, Control, Priya            OK
bangalore.koramangala            Control                          OK
bangalore.koramangala.food       Control, Meena, Ravi             OK
bangalore.koramangala.ride       Arjun, Control, Ravi             (not in D1 table)
bangalore.whitefield.food        Control, Meena                   (not in D1 table)
bangalore.whitefield.ride        Arjun, Control                   OK
chennai.tnagar.food              Karthik                          OK
chennai.tnagar.ride              dropped                          (not in D1 table)
chennai.whitefield.food          dropped                          (not in D1 table)
chennai.whitefield.ride          dropped                          (not in D1 table)

D1 key 5 (chennai.whitefield.ride) published: True | received by: nobody

# D3

In [8]:
## ACK & NACK

!git pull -q
!grep -n "BAD" stream/dispatcher.py

34:    print(f"published key={routing_key} id={message['id']}" + (" BAD(no zone)" if bad else ""), flush=True)


In [9]:
import subprocess, time, os
os.makedirs("logs", exist_ok=True)

drivers = [("Ravi", "bangalore.koramangala.*"), ("Priya", "bangalore.indiranagar.*"),
           ("Arjun", "bangalore.*.ride"), ("Meena", "bangalore.*.food"),
           ("Karthik", "chennai.tnagar.food"), ("Control", "bangalore.#")]

procs = []
for name, pattern in drivers:
    f = open(f"logs/{name}.log", "w")
    procs.append(subprocess.Popen(["python", "-u", "stream/driver.py", name, pattern],
                                  stdout=f, stderr=subprocess.STDOUT))
time.sleep(4)                                       # let every driver bind its queue

disp = subprocess.Popen(["python", "-u", "stream/dispatcher.py"],
                        stdout=open("logs/dispatcher.log", "w"), stderr=subprocess.STDOUT)

time.sleep(6)
# one extra message for D1 question 6 (a two-word key the random dispatcher never makes)
subprocess.run(["python", "-u", "stream/dispatcher.py", "bangalore.koramangala"],
               stdout=open("logs/oneshot.log", "w"), stderr=subprocess.STDOUT)

time.sleep(114)                                     # about two minutes in total
disp.terminate()
time.sleep(3)
for p in procs:
    p.terminate()
print("done")

done


In [10]:
import re, glob
bad_ids = re.findall(r"published key=(\S+) id=(\w+) BAD", open("logs/dispatcher.log").read())
print("Bad messages published:", bad_ids, "\n")

for key, mid in bad_ids:
    print(f"--- id {mid} (key {key})")
    for f in sorted(glob.glob("logs/*.log")):
        name = f.split("/")[-1][:-4]
        if name in ("dispatcher", "oneshot"):
            continue
        for line in open(f):
            if f"id={mid}" in line:
                print("   ", line.strip())

print("\nGood messages processed per driver:")
for f in sorted(glob.glob("logs/*.log")):
    name = f.split("/")[-1][:-4]
    if name in ("dispatcher", "oneshot"):
        continue
    text = open(f).read()
    print(f"   {name:8s} processed {len(re.findall(r'\] key=', text))}, rejected {text.count('REJECTED')}")

Bad messages published: [] 


Good messages processed per driver:
   Arjun    processed 0, rejected 0
   Control  processed 1, rejected 0
   Karthik  processed 0, rejected 0
   Meena    processed 0, rejected 0
   Priya    processed 0, rejected 0
   Ravi     processed 0, rejected 0
